In [1]:
import os
os.getcwd()

'/Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM'

In [2]:
from InterOptimus.agents.llm_iomaker_job import BuildConfig, build_iomaker_flow_from_prompt
from qtoolkit.core.data_objects import QResources

/opt/anaconda3/envs/3.12/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
#!/usr/bin/env python3
"""
Demo: LLM IOMaker with remote submission.

This script demonstrates how to use BuildConfig to automatically submit
generated jobflow jobs to a remote server with VASP calculation.

Key features:
- Uses default MPRelaxSet/MPStaticSet when vasp_relax_settings/vasp_static_settings are not provided
- Automatic remote submission via SSH
- Passphrase support for SSH keys
"""

from InterOptimus.agents.llm_iomaker_job import BuildConfig, build_iomaker_flow_from_prompt
from qtoolkit.core.data_objects import QResources
import os


def mlip_resources():
    """Return QResources for MLIP jobs."""
    return QResources(
        nodes=1,
        processes_per_node=1,
        scheduler_kwargs={"partition": "standard", "cpus_per_task": 10},
    )


def vasp_resources():
    """Return QResources for VASP jobs."""
    return QResources(
        nodes=1,
        processes_per_node=64,
        scheduler_kwargs={"partition": "standard"},
    )


def main():
    # Configure LLM API
    API_SECRET_KEY = "sk-zk29a8909a9badfb2973536dfd5a1bf41c7696742c764c35"
    BASE_URL = "https://api.zhizengzeng.com/v1/"

    # Create BuildConfig with remote submission enabled
    cfg = BuildConfig(
        # LLM configuration
        api_key=API_SECRET_KEY,
        base_url=BASE_URL,
        model="gpt-3.5-turbo",
        
        # Materials Project (optional, only needed if using MP IDs)
        mp_api_key="LBPWGcTye2eanNBb7dtmTp2deXR4aF9E",
        
        # Input files (defaults to film.cif and substrate.cif in current directory)
        film_cif="film.cif",
        substrate_cif="substrate.cif",
        
        # Output
        output_flow_json="io_flow.json",
        print_settings=True,
        
        # Jobflow resources and workers
        mlip_resources=mlip_resources,
        mlip_worker="std_worker",
        ckpt_path="/home/xys/check_points/orb-v3-conservative-inf-omat-20250404.ckpt",
        
        # VASP configuration (required if do_vasp=True in LLM output)
        vasp_resources=vasp_resources,  # Required if do_vasp=True
        vasp_worker="vasp_worker",  # Required if do_vasp=True
        
        # VASP settings: Not provided - will use default MPRelaxSet/MPStaticSet
        # vasp_relax_settings=None,  # Default: uses MPRelaxSet defaults
        # vasp_static_settings=None,  # Default: uses MPStaticSet defaults
        
        # Remote submission configuration
        submit_to_remote=True,  # Enable automatic remote submission
        remote_host="xys@10.103.65.21",
        remote_identity_file="~/.ssh/id_ed25519",
        remote_workdir="~/io_runs/si_sic_run1",
        remote_python="python",
        remote_pre_cmd="source ~/.bashrc && conda activate atomate2",
        # Use environment variable for passphrase in production!
        remote_passphrase=os.getenv("SSH_PASSPHRASE", "jinlhr542"),
        remote_use_paramiko=False,  # Use pexpect (recommended)
        remote_debug=False,  # Set to True for verbose output
    )

    # Build jobflow from natural language prompt
    user_prompt = "建立双界面模型，进行VASP模拟，不要作梯度下降"
    
    print("=" * 80)
    print("Building IOMaker jobflow from prompt...")
    print(f"Prompt: {user_prompt}")
    print("=" * 80)
    
    try:
        result = build_iomaker_flow_from_prompt(user_prompt, cfg)
        
        print("\n✅ Jobflow generation completed!")
        print(f"📄 Flow JSON: {result['flow_json_path']}")
        
        # Check remote submission result
        if "remote_submission" in result:
            submit_result = result["remote_submission"]
            if submit_result["success"]:
                print(f"\n✅ Remote submission successful!")
                if submit_result.get("job_id"):
                    print(f"📋 Job ID: {submit_result['job_id']}")
            else:
                print(f"\n❌ Remote submission failed:")
                print(f"   Error: {submit_result.get('error', 'unknown')}")
                print(f"   Details: {submit_result.get('stderr', '')}")
        
        if "remote_submission_error" in result:
            print(f"\n⚠️  Remote submission error: {result['remote_submission_error']}")
            
    except ValueError as e:
        print(f"\n❌ Validation error: {e}")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

Building IOMaker jobflow from prompt...
Prompt: 建立双界面模型，进行VASP模拟，不要作梯度下降


/Users/jason/Downloads/pymatgen-master/src/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]



✅ InterOptimus IOMaker job generated
📄 Flow JSON: /Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM/io_flow.json
📦 Structures source: local_cif
🧱 film.cif: /Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM/film.cif
🧱 substrate.cif: /Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM/substrate.cif

⚙️  Jobflow configuration:
   MLIP worker: std_worker
   MLIP resources: ✅ Set
   VASP worker: vasp_worker
   VASP resources: ✅ Set

🧪 VASP input settings:
   - vasp_relax_settings: (default) pymatgen.io.vasp.sets.MPRelaxSet
   - vasp_static_settings: (default) pymatgen.io.vasp.sets.MPStaticSet

🌐 Remote submission configuration:
   Host: xys@10.103.65.21
   Remote workdir: ~/io_runs/si_sic_run1
   Python: python
   Pre-command: source ~/.bashrc && conda activate atomate2

🔧 Parameter settings (LLM output normalized):
{
  "name": "IO_llm",
  "mode": "without_vacuum",
  "inputs": {
    "type": "local_cif",
    "film_cif": "film.cif",
    "substrate_cif": "substrate.cif"
  },
 

In [1]:
import re, json, hashlib
from datetime import datetime

def _slug(s: str, maxlen: int = 32) -> str:
    s = (s or "NA")
    s = re.sub(r"\s+", "-", s.strip())
    s = re.sub(r"[^A-Za-z0-9._-]+", "", s)
    return s[:maxlen] or "NA"

def make_iomaker_name(
    film: str,
    substrate: str,
    mode: str,          # "with_vacuum" / "without_vacuum"
    do_vasp: bool,
    mlip_calc: str,     # e.g. "orb-models"
    key_settings: dict, # 只放你认为会改变结果的关键参数
) -> str:
    mode_tag = "double" if mode == "without_vacuum" else "vac"
    calc_tag = {"orb-models":"orb", "sevenn":"sevenn", "dpa":"dpa"}.get(mlip_calc, _slug(mlip_calc, 12))
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    blob = json.dumps(key_settings, sort_keys=True, ensure_ascii=False)
    h8 = hashlib.sha1(blob.encode("utf-8")).hexdigest()[:8]
    return f"IO_{_slug(film)}-{_slug(substrate)}_m-{mode_tag}_v{1 if do_vasp else 0}_mlip-{calc_tag}_{ts}_{h8}"

In [3]:
make_iomaker_name('Li', 'Si', 'double', True, 'orb-models', {})

'IO_Li-Si_m-vac_v1_mlip-orb_20260115-145902_bf21a9e8'